# Adaptive Therapy Modeling: ODE vs. PDE Frameworks

## Overview
This notebook implements a comparative analysis of tumor resistance dynamics using two mathematical frameworks:
1.  **ODE (Ordinary Differential Equations):** Models the tumor as a well-mixed liquid. Fast and good for parameter estimation.
2.  **PDE (Partial Differential Equations):** Models the tumor as a spatial entity with diffusion. Used to study spatial constraints and competition.

### The Biological Model
We track two populations:
* $S(t)$: Drug-Sensitive cancer cells.
* $R(t)$: Drug-Resistant cancer cells.

The governing equation (Reaction-Diffusion) is:
$$\frac{\partial S}{\partial t} = \nabla \cdot (D_S \nabla S) + r_S S (1 - \frac{S+R}{K}) - d_S(t) S$$
$$\frac{\partial R}{\partial t} = \nabla \cdot (D_R \nabla R) + r_R R (1 - \frac{S+R}{K}) - d_R(t) R$$

---

In [ ]:
# Import Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize

# FEniCSx Imports (for PDE)
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem import petsc as fem_petsc
import ufl
import basix.ufl

# Visualization Settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading
We load the patient data (LiquidCNA estimates) to fit our models. 
*Note: Ensure your data paths match the variables below.*

In [ ]:
DATA_PATH = "data/liquidCNA_results/Subclonal_ratio_estimates.extended.txt"
SAMPLE_LIST = "data/OV_patientDNA_sampleList.txt"
TARGET_PATIENT = "UP0018"  # The 'Gold Standard' patient

def load_patient_data(patient_id):
    # Simplified loader logic
    try:
        df = pd.read_csv(DATA_PATH, sep='\t')
        # Filter for patient
        df = df[df['Patient'] == patient_id].copy()
        
        # Basic cleaning
        df['Time'] = pd.to_numeric(df['Time'], errors='coerce')
        df['CA125'] = pd.to_numeric(df['CA125'], errors='coerce')
        df = df.dropna(subset=['Time', 'ratio', 'CA125'])
        
        # Convert days to months if needed (assuming raw is Days)
        # Adjust this based on your actual data format
        if df['Time'].max() > 100: 
            df['Time'] = df['Time'] / 30.0
            
        return df.sort_values('Time')
    except Exception as e:
        print(f"Error loading data: {e}")
        return pd.DataFrame()

data = load_patient_data(TARGET_PATIENT)
print(f"Loaded {len(data)} timepoints for {TARGET_PATIENT}")
display(data.head())

## 2. ODE Model Fitting (Temporal)
First, we ignore space ($D=0$) to estimate the biological parameters ($a_S, a_R, d_S$).
We minimize the error between the model predictions and the clinical observations (Ratio and CA125).

In [ ]:
def ode_system(t, y, params):
    S, R = y
    aS, aR, dS, dR, K = params
    u = 1.0 # Constant treatment
    
    N = S + R
    dSdt = aS * S * (1 - N/K) - u * dS * S
    dRdt = aR * R * (1 - N/K) - u * dR * R
    return [dSdt, dRdt]

def objective_function(params_log, data):
    # Unwrap parameters (working in log space to enforce positivity)
    params = np.exp(params_log)
    aS, aR, dS, dR, K, N0_scale = params
    
    # Initial Conditions from data
    r0 = data['ratio'].iloc[0]
    # CA125 ~ N * gamma. We assume gamma=1 for optimization simplicity here, or fold into N0
    S0 = (1 - r0) * N0_scale
    R0 = r0 * N0_scale
    
    # Solve ODE
    t_span = [data['Time'].min(), data['Time'].max()]
    sol = solve_ivp(ode_system, t_span, [S0, R0], args=([aS, aR, dS, dR, K],), 
                    t_eval=data['Time'].values, method='LSODA')
    
    if not sol.success: return 1e9
    
    # Calculate Observables
    S_pred, R_pred = sol.y
    N_pred = S_pred + R_pred
    ratio_pred = R_pred / (N_pred + 1e-9)
    
    # Error (Weighted sum of squares)
    err_ratio = np.mean((ratio_pred - data['ratio'].values)**2)
    err_ca = np.mean((np.log(N_pred) - np.log(data['CA125'].values))**2)
    
    return err_ratio + 0.1 * err_ca # Weight ratio more heavily

# Run Optimization
initial_guess = np.log([0.5, 0.3, 0.4, 0.01, 100000, data['CA125'].iloc[0]])
res = minimize(objective_function, initial_guess, args=(data,), method='Nelder-Mead')

fitted_params = np.exp(res.x)
print("Fitted Parameters:")
print(f"aS: {fitted_params[0]:.3f}, aR: {fitted_params[1]:.3f}")
print(f"dS: {fitted_params[2]:.3f}, dR: {fitted_params[3]:.3f}")
print(f"K:  {fitted_params[4]:.0f}")

## 3. PDE Simulation (Spatio-Temporal)
Now we take the fitted parameters and inject them into the FEniCSx solver. We add diffusion coefficients ($D_S, D_R$) to see how spatial constraints affect the outcome.

In [ ]:
def run_pde_simulation(params, data, n_cells=100):
    aS, aR, dS, dR, K, N0_scale = params
    DS, DR = 0.01, 0.01 # Diffusion coefficients
    
    # Mesh (1D Line)
    comm = MPI.COMM_WORLD
    domain = mesh.create_interval(comm, n_cells, [0.0, 1.0])
    V = fem.functionspace(domain, ("Lagrange", 1))
    
    # Functions
    S = fem.Function(V); R = fem.Function(V)
    S_prev = fem.Function(V); R_prev = fem.Function(V)
    
    # ICs
    r0 = data['ratio'].iloc[0]
    S.interpolate(lambda x: np.full(x.shape[1], (1-r0)*N0_scale))
    R.interpolate(lambda x: np.full(x.shape[1], r0*N0_scale))
    S_prev.x.array[:] = S.x.array[:]
    R_prev.x.array[:] = R.x.array[:]
    
    # Variational Form (Implicit Euler)
    dt = 0.01
    u_trial, v_test = ufl.TrialFunction(V), ufl.TestFunction(V)
    
    # Diffusion Terms
    F_S = (ufl.inner(u_trial, v_test)*ufl.dx 
           + dt*DS*ufl.inner(ufl.grad(u_trial), ufl.grad(v_test))*ufl.dx 
           - ufl.inner(S_prev, v_test)*ufl.dx)
    F_R = (ufl.inner(u_trial, v_test)*ufl.dx 
           + dt*DR*ufl.inner(ufl.grad(u_trial), ufl.grad(v_test))*ufl.dx 
           - ufl.inner(R_prev, v_test)*ufl.dx)
           
    # Assemble Matrices (Constant in time)
    prob_S = fem_petsc.LinearProblem(ufl.lhs(F_S), ufl.rhs(F_S), u=S, petsc_options={"ksp_type": "cg"})
    prob_R = fem_petsc.LinearProblem(ufl.lhs(F_R), ufl.rhs(F_R), u=R, petsc_options={"ksp_type": "cg"})
    
    # Time Loop
    t_max = data['Time'].max()
    times = np.arange(0, t_max + dt, dt)
    
    history = {'t': [], 'ratio': [], 'N': [], 'S_field': [], 'R_field': []}
    
    for t in times:
        # Reaction (Explicit)
        s_vec, r_vec = S.x.array, R.x.array
        n_vec = s_vec + r_vec
        
        growth_s = aS * s_vec * (1 - n_vec/K)
        growth_r = aR * r_vec * (1 - n_vec/K)
        death_s = dS * s_vec
        death_r = dR * r_vec
        
        S_prev.x.array[:] = s_vec + dt * (growth_s - death_s)
        R_prev.x.array[:] = r_vec + dt * (growth_r - death_r)
        
        # Diffusion (Implicit)
        prob_S.solve()
        prob_R.solve()
        
        # Record
        total_S = np.mean(S.x.array)
        total_R = np.mean(R.x.array)
        history['t'].append(t)
        history['N'].append(total_S + total_R)
        history['ratio'].append(total_R / (total_S + total_R + 1e-9))
        history['S_field'].append(S.x.array.copy())
        history['R_field'].append(R.x.array.copy())
        
    return history

# Run PDE
pde_res = run_pde_simulation(fitted_params, data)

## 4. Visualization: Space-Time Heatmaps
Here we visualize the evolution of the tumor density and resistance across the 1D domain.

In [ ]:
# Convert History to Arrays
S_mat = np.array(pde_res['S_field'])
R_mat = np.array(pde_res['R_field'])
Total = S_mat + R_mat
Ratio = R_mat / (Total + 1e-9)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Total Density Heatmap
im1 = ax[0].imshow(Total, aspect='auto', origin='lower', cmap='viridis', 
                   extent=[0, 1, 0, pde_res['t'][-1]])
ax[0].set_title(f"Patient {TARGET_PATIENT}: Total Tumor Density")
ax[0].set_xlabel("Space (x)")
ax[0].set_ylabel("Time (Months)")
plt.colorbar(im1, ax=ax[0], label="Density")

# Resistance Fraction Heatmap
im2 = ax[1].imshow(Ratio, aspect='auto', origin='lower', cmap='inferno', vmin=0, vmax=1,
                   extent=[0, 1, 0, pde_res['t'][-1]])
ax[1].set_title("Evolution of Resistance")
ax[1].set_xlabel("Space (x)")
plt.colorbar(im2, ax=ax[1], label="Resistant Fraction")

plt.tight_layout()
plt.show()

## 5. Model Comparison
Finally, we compare the PDE trajectory against the clinical data to validate the fit.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Ratio Plot
ax[0].plot(data['Time'], data['ratio'], 'ko', label='Clinical Data', markersize=8)
ax[0].plot(pde_res['t'], pde_res['ratio'], 'r-', linewidth=3, label='PDE Prediction')
ax[0].set_title("Resistant Fraction")
ax[0].set_ylim(0, 1)
ax[0].legend()

# Tumor Size Plot (Log Scale for CA125 proxy)
# Note: PDE calculates N, Data is CA125. We scale PDE N to match mean CA125 magnitude
scale_factor = np.mean(data['CA125']) / np.mean(pde_res['N'])

ax[1].plot(data['Time'], np.log(data['CA125']), 'ko', label='Clinical Data', markersize=8)
ax[1].plot(pde_res['t'], np.log(np.array(pde_res['N']) * scale_factor), 'b-', linewidth=3, label='PDE Prediction')
ax[1].set_title("Tumor Burden (Log Scale)")
ax[1].legend()

plt.show()